## SONAR Text Embedding Demo

This notebook now follows the repository logic: encode words and definitions with SONAR in batches, inspect the embedding spaces with PCA, and compute embedding-space metrics without decoding text back into words or definitions.

In [1]:
# Config plotly to work in Jupyter Notebook
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'
import numpy as np
from tqdm import tqdm as track

In [2]:
import torch
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = torch.device(DEVICE)
torch.set_grad_enabled(False)
print(DEVICE)

# load encoder and decoder
text2vec = TextToEmbeddingModelPipeline(
    encoder="text_sonar_basic_encoder",
    tokenizer="text_sonar_basic_encoder",
    device=DEVICE,
 )

Output()

cpu


# Calc with dictionary dataset

In [4]:
from datasets import load_dataset
import numpy as np
from sklearn.decomposition import PCA

In [3]:
EMBEDDING_BATCH_SIZE = 64
WORD_MAX_SEQ_LEN = 64
DEFINITION_MAX_SEQ_LEN = 512

In [5]:
ds = load_dataset("npvinHnivqn/EnglishDictionary")

split = ds["train"].train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
test_ds = split["test"]

len(train_ds), len(test_ds)

(100440, 11161)

In [ ]:
def clean_definition(definition):
    definition = "" if definition is None else str(definition).strip()
    if "." not in definition:
        return definition
    parts = [part.strip() for part in definition.split(".") if part.strip()]
    if not parts:
        return definition
    return max(parts, key=len)

def encode_texts(texts, max_seq_len):
    if len(texts) == 0:
        return np.zeros((0, 1024), dtype=np.float32)

    batches = []
    for start in track(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Encoding batches"):
        batch = texts[start : start + EMBEDDING_BATCH_SIZE]
        embedding = text2vec.predict(
            batch,
            source_lang="eng_Latn",
            max_seq_len=max_seq_len,
        )
        batches.append(embedding.detach().cpu().numpy())
    return np.concatenate(batches, axis=0)

In [ ]:
words = [str(word).strip() for word in test_ds["word"]]
defs = [clean_definition(definition) for definition in test_ds["definition"]]

word_embeddings = encode_texts(words, WORD_MAX_SEQ_LEN)
def_embeddings = encode_texts(defs, DEFINITION_MAX_SEQ_LEN)

word_embeddings.shape, def_embeddings.shape

In [ ]:
# Save the embeddings to disk for later use
np.save("data/word_embeddings.npy", word_embeddings)
np.save("data/def_embeddings.npy", def_embeddings)


In [ ]:
# Load the embeddings from disk (if needed)
word_embeddings = np.load("data/word_embeddings.npy")
def_embeddings = np.load("data/def_embeddings.npy")
words = np.load("data/words.npy")
defs = np.load("data/defs.npy")

In [ ]:
# Joint PCA visualization of the embeddings in 2D

all_embeddings = np.concatenate([word_embeddings, def_embeddings], axis=0)
pca = PCA(n_components=2)
all_emb_2d = pca.fit_transform(all_embeddings)

split_idx = len(word_embeddings)
word_emb_2d = all_emb_2d[:split_idx]
def_emb_2d = all_emb_2d[split_idx:]

fig = px.scatter(
    x=word_emb_2d[:, 0],
    y=word_emb_2d[:, 1],
    hover_name=words,
    color_discrete_sequence=["blue"],
    title="Joint PCA of Word and Definition Embeddings (2D)",
)
fig.add_scatter(
    x=def_emb_2d[:, 0],
    y=def_emb_2d[:, 1],
    hovertext=defs,
    mode="markers",
    marker=dict(color="red"),
    name="Definitions",
)
fig.show()

In [ ]:
# Joint PCA visualization of the embeddings in 3D

all_embeddings = np.concatenate([word_embeddings, def_embeddings], axis=0)
pca = PCA(n_components=3)
all_emb_3d = pca.fit_transform(all_embeddings)

split_idx = len(word_embeddings)
word_emb_3d = all_emb_3d[:split_idx]
def_emb_3d = all_emb_3d[split_idx:]

fig = px.scatter_3d(
    x=word_emb_3d[:, 0],
    y=word_emb_3d[:, 1],
    z=word_emb_3d[:, 2],
    color_discrete_sequence=["blue"],
    title="Joint PCA of Word and Definition Embeddings (3D)",
)
fig.add_scatter3d(
    x=def_emb_3d[:, 0],
    y=def_emb_3d[:, 1],
    z=def_emb_3d[:, 2],
    mode="markers",
    marker=dict(color="red"),
    name="Definitions",
)
fig.show()

In [ ]:
from sonar.inference_pipelines.text import EmbeddingToTextModelPipeline
vec2text = EmbeddingToTextModelPipeline(
    decoder="text_sonar_basic_decoder",
    tokenizer="text_sonar_basic_encoder",
    device=DEVICE,
 )

In [ ]:
from nltk.metrics import edit_distance as levenshtein_distance

def row_cosine(a, b):
    a_norm = np.linalg.norm(a, axis=1)
    b_norm = np.linalg.norm(b, axis=1)
    denom = np.where((a_norm * b_norm) == 0.0, 1.0, a_norm * b_norm)
    return np.sum(a * b, axis=1) / denom

reconstructed_words = []
reconstructed_defs = []

for start in track(range(0, len(words), EMBEDDING_BATCH_SIZE), desc="Decoding words"):
    batch = word_embeddings[start : start + EMBEDDING_BATCH_SIZE]
    reconstructed_batch = vec2text.predict(
        torch.from_numpy(batch).to(DEVICE),
        target_lang="eng_Latn",
        max_seq_len=WORD_MAX_SEQ_LEN,
    )
    reconstructed_words.extend(reconstructed_batch)

for start in track(range(0, len(defs), EMBEDDING_BATCH_SIZE), desc="Decoding definitions"):
    batch = def_embeddings[start : start + EMBEDDING_BATCH_SIZE]
    reconstructed_batch = vec2text.predict(
        torch.from_numpy(batch).to(DEVICE),
        target_lang="eng_Latn",
        max_seq_len=DEFINITION_MAX_SEQ_LEN,
    )
    reconstructed_defs.extend(reconstructed_batch)

reconstructed_word_embeddings = encode_texts(reconstructed_words, WORD_MAX_SEQ_LEN)
reconstructed_def_embeddings = encode_texts(reconstructed_defs, DEFINITION_MAX_SEQ_LEN)

word_reconstruction_cosine = row_cosine(word_embeddings, reconstructed_word_embeddings)
def_reconstruction_cosine = row_cosine(def_embeddings, reconstructed_def_embeddings)
word_reconstruction_l2 = np.linalg.norm(word_embeddings - reconstructed_word_embeddings, axis=1)
def_reconstruction_l2 = np.linalg.norm(def_embeddings - reconstructed_def_embeddings, axis=1)

word_edit_distances = [
    levenshtein_distance(original.lower(), reconstructed.lower())
    for original, reconstructed in zip(words, reconstructed_words)
 ]
def_edit_distances = [
    levenshtein_distance(original.lower(), reconstructed.lower())
    for original, reconstructed in zip(defs, reconstructed_defs)
 ]

print(f"Average word reconstruction cosine: {word_reconstruction_cosine.mean():.4f}")
print(f"Average definition reconstruction cosine: {def_reconstruction_cosine.mean():.4f}")
print(f"Average word reconstruction L2: {word_reconstruction_l2.mean():.4f}")
print(f"Average definition reconstruction L2: {def_reconstruction_l2.mean():.4f}")
print(f"Average word Levenshtein distance: {np.mean(word_edit_distances):.4f}")
print(f"Average definition Levenshtein distance: {np.mean(def_edit_distances):.4f}")

for i in range(min(10, len(words))):
    print("-" * 100)
    print(f"Original word: {words[i]}")
    print(f"Reconstructed word: {reconstructed_words[i]}")
    print(f"Word cosine: {word_reconstruction_cosine[i]:.4f}")
    print(f"Word L2: {word_reconstruction_l2[i]:.4f}")
    print(f"Word Levenshtein: {word_edit_distances[i]}")
    print()
    print(f"Original definition: {defs[i]}")
    print(f"Reconstructed definition: {reconstructed_defs[i]}")
    print(f"Definition cosine: {def_reconstruction_cosine[i]:.4f}")
    print(f"Definition L2: {def_reconstruction_l2[i]:.4f}")
    print(f"Definition Levenshtein: {def_edit_distances[i]}")
    print()